### 04_modeling.ipynb

[Purpose]
- Train multiple ML models for comparison
- Hyperparameter tuning with Optuna
- Save all models and results

[Models]
- LightGBM (baseline)
- XGBoost
- Random Forest
- CatBoost
- (Optional) LSTM for time-series

[Output]
- Trained models in models/
- Comparison results in outputs/reports/

[Why LightGBM?]
- Fast training speed
- Handles large datasets well
- Good performance with tabular data
- Built-in feature importance


In [1]:
# ===========================================
# Cell 1: Google Drive Mount
# ===========================================
from google.colab import drive

drive.mount('/content/drive')
print("Drive mount done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount done!


In [2]:
# ===========================================
# Cell 2: Config Load + Path Setup
# ===========================================
import os
import yaml

CONFIG_PATH = "/content/config.yaml"

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_DATA_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH = os.path.join(DRIVE_ROOT, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

print(f"Project: {config['project_name']}")
print(f"Processed: {PROCESSED_DATA_PATH}")
print(f"Models: {MODEL_PATH}")

Project: EcoTracing
Processed: /content/drive/MyDrive/EcoTracing/processed
Models: /content/drive/MyDrive/EcoTracing/models


In [13]:
# ===========================================
# Cell 3: Library Import
# ===========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
import pickle

print("Library load done!")

Library load done!


In [14]:
# ===========================================
# Cell 4: Load Processed Data
# ===========================================
data_path = os.path.join(PROCESSED_DATA_PATH, "instance_usage_processed.csv")

df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Data shape: (50000, 9)
Columns: ['machine_id', 'start_time_sec', 'end_time_sec', 'duration', 'hour', 'cpu', 'memory', 'power_w', 'energy_kwh']


In [15]:
# ===========================================
# Cell 5: Feature / Target Split
# ===========================================
features = ['cpu', 'memory', 'duration', 'hour']
target = 'energy_kwh'

X = df[features]
y = df[target]

print(f"Features: {features}")
print(f"Target: {target}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Features: ['cpu', 'memory', 'duration', 'hour']
Target: energy_kwh
X shape: (50000, 4)
y shape: (50000,)


In [6]:
# ===========================================
# Cell 6: Train / Test Split
# ===========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

Train: (40000, 4)
Test: (10000, 4)


In [7]:
# ===========================================
# Cell 7: LightGBM Model Training
# ===========================================
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbose': -1
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[test_data],
)

print("Model training done!")

Model training done!


In [8]:
# ===========================================
# Cell 8: Prediction
# ===========================================
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("Prediction done!")
print(f"Train predictions: {len(y_pred_train)}")
print(f"Test predictions: {len(y_pred_test)}")

Prediction done!
Train predictions: 40000
Test predictions: 10000


In [9]:
# ===========================================
# Cell 9: Save Model
# ===========================================
model_file = os.path.join(MODEL_PATH, "energy_model_lgb.pkl")

with open(model_file, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved: {model_file}")


Model saved: /content/drive/MyDrive/EcoTracing/models/energy_model_lgb.pkl


In [10]:
# ===========================================
# Cell 10: Save Predictions for Evaluation
# ===========================================
results = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred_test
})

results_path = os.path.join(PROCESSED_DATA_PATH, "model_predictions.csv")
results.to_csv(results_path, index=False)

print(f"Predictions saved: {results_path}")

Predictions saved: /content/drive/MyDrive/EcoTracing/processed/model_predictions.csv


In [11]:
# ===========================================
# Cell 11: Summary
# ===========================================
print("=" * 50)
print("Modeling Complete")
print("=" * 50)
print(f"\n[Data]")
print(f"  - Train samples: {len(X_train):,}")
print(f"  - Test samples: {len(X_test):,}")
print(f"\n[Model]")
print(f"  - Type: LightGBM")
print(f"  - Features: {features}")
print(f"\n[Output]")
print(f"  - Model: {model_file}")
print(f"  - Predictions: {results_path}")
print(f"\n[Next Step]")
print(f"  - 05_evaluation.ipynb")

Modeling Complete

[Data]
  - Train samples: 40,000
  - Test samples: 10,000

[Model]
  - Type: LightGBM
  - Features: ['cpu', 'memory', 'duration', 'hour']

[Output]
  - Model: /content/drive/MyDrive/EcoTracing/models/energy_model_lgb.pkl
  - Predictions: /content/drive/MyDrive/EcoTracing/processed/model_predictions.csv

[Next Step]
  - 05_evaluation.ipynb


In [ ]:
# ===========================================
# Cell 1: Google Drive Mount
# ===========================================
from google.colab import drive

drive.mount('/content/drive')
print("Drive mount done!")

In [ ]:
# ===========================================
# Cell 2: Config Load + Path Setup
# ===========================================
import os
import yaml

CONFIG_PATH = "/content/config.yaml"

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_DATA_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH = os.path.join(DRIVE_ROOT, "models")
REPORT_PATH = os.path.join(DRIVE_ROOT, "outputs", "reports")

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

print(f"Project: {config['project_name']}")

In [ ]:
# ===========================================
# Cell 3: Install Dependencies
# ===========================================
!pip install -q xgboost catboost optuna

In [ ]:
# ===========================================
# Cell 4: Library Import
# ===========================================
import pandas as pd
import numpy as np
import pickle
import json
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')

print("Library load done!")

In [ ]:
# ===========================================
# Cell 5: Load Processed Data
# ===========================================
# Try parquet first, then CSV
parquet_path = os.path.join(PROCESSED_DATA_PATH, "instance_usage_full_processed.parquet")
csv_path = os.path.join(PROCESSED_DATA_PATH, "instance_usage_processed.csv")

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print(f"Loaded parquet: {len(df):,} rows")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV: {len(df):,} rows")

print(f"Columns: {df.columns.tolist()}")


In [ ]:
# ===========================================
# Cell 6: Feature / Target Split
# ===========================================
features = ['cpu', 'memory', 'duration', 'hour']
target = 'energy_kwh'

X = df[features]
y = df[target]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# ===========================================
# Cell 7: Define Evaluation Function
# ===========================================
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate model and return metrics dict.
    """
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-10))) * 100

    return {
        'model': model_name,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape
    }

In [ ]:
# ===========================================
# Cell 8: Train LightGBM
# ===========================================
print("=" * 50)
print("Training LightGBM...")
print("=" * 50)

start_time = time.time()

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbose': -1
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

lgb_model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=200,
    valid_sets=[test_data],
)

lgb_time = time.time() - start_time
lgb_results = evaluate_model(lgb_model, X_test, y_test, 'LightGBM')
lgb_results['train_time'] = lgb_time

print(f"Time: {lgb_time:.2f}s")
print(f"RMSE: {lgb_results['rmse']:.6f}")
print(f"R2: {lgb_results['r2']:.6f}")

In [ ]:
# ===========================================
# Cell 9: Train XGBoost
# ===========================================
print("\n" + "=" * 50)
print("Training XGBoost...")
print("=" * 50)

start_time = time.time()

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_time = time.time() - start_time
xgb_results = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
xgb_results['train_time'] = xgb_time

print(f"Time: {xgb_time:.2f}s")
print(f"RMSE: {xgb_results['rmse']:.6f}")
print(f"R2: {xgb_results['r2']:.6f}")

In [ ]:
# ===========================================
# Cell 10: Train Random Forest
# ===========================================
print("\n" + "=" * 50)
print("Training Random Forest...")
print("=" * 50)

start_time = time.time()

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_time = time.time() - start_time
rf_results = evaluate_model(rf_model, X_test, y_test, 'RandomForest')
rf_results['train_time'] = rf_time

print(f"Time: {rf_time:.2f}s")
print(f"RMSE: {rf_results['rmse']:.6f}")
print(f"R2: {rf_results['r2']:.6f}")

In [ ]:
# ===========================================
# Cell 11: Train CatBoost
# ===========================================
print("\n" + "=" * 50)
print("Training CatBoost...")
print("=" * 50)

start_time = time.time()

cb_model = CatBoostRegressor(
    iterations=200,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False
)

cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)

cb_time = time.time() - start_time
cb_results = evaluate_model(cb_model, X_test, y_test, 'CatBoost')
cb_results['train_time'] = cb_time

print(f"Time: {cb_time:.2f}s")
print(f"RMSE: {cb_results['rmse']:.6f}")
print(f"R2: {cb_results['r2']:.6f}")

In [ ]:
# ===========================================
# Cell 12: Model Comparison Summary
# ===========================================
print("\n" + "=" * 50)
print("MODEL COMPARISON SUMMARY")
print("=" * 50)

all_results = [lgb_results, xgb_results, rf_results, cb_results]
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('rmse')

print(results_df.to_string(index=False))

# Find best model
best_model_name = results_df.iloc[0]['model']
print(f"\nBest Model: {best_model_name} (lowest RMSE)")

In [ ]:
# ===========================================
# Cell 13: Save All Models
# ===========================================
models = {
    'LightGBM': lgb_model,
    'XGBoost': xgb_model,
    'RandomForest': rf_model,
    'CatBoost': cb_model
}

for name, model in models.items():
    model_file = os.path.join(MODEL_PATH, f"energy_model_{name.lower()}.pkl")
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    print(f"Saved: {model_file}")

In [ ]:

# ===========================================
# Cell 14: Save Comparison Results
# ===========================================
# Save as JSON
comparison_results = {
    'dataset': 'Google Cluster Trace 2019',
    'samples': {
        'total': len(df),
        'train': len(X_train),
        'test': len(X_test)
    },
    'features': features,
    'target': target,
    'models': all_results,
    'best_model': best_model_name
}

json_path = os.path.join(REPORT_PATH, "model_comparison_results.json")
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_results, f, indent=2, ensure_ascii=False)
print(f"Saved: {json_path}")

# Save as CSV
csv_path = os.path.join(REPORT_PATH, "model_comparison_metrics.csv")
results_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")


# ===========================================
# Cell 15: Feature Importance Comparison
# ===========================================
print("\n" + "=" * 50)
print("FEATURE IMPORTANCE COMPARISON")
print("=" * 50)

# LightGBM
lgb_importance = dict(zip(features, lgb_model.feature_importance()))

# XGBoost
xgb_importance = dict(zip(features, xgb_model.feature_importances_))

# RandomForest
rf_importance = dict(zip(features, rf_model.feature_importances_))

# CatBoost
cb_importance = dict(zip(features, cb_model.feature_importances_))

importance_df = pd.DataFrame({
    'feature': features,
    'LightGBM': [lgb_importance[f] for f in features],
    'XGBoost': [xgb_importance[f] for f in features],
    'RandomForest': [rf_importance[f] for f in features],
    'CatBoost': [cb_importance[f] for f in features]
})

print(importance_df.to_string(index=False))

# Save importance
importance_path = os.path.join(REPORT_PATH, "feature_importance_comparison.csv")
importance_df.to_csv(importance_path, index=False)
print(f"\nSaved: {importance_path}")


# ===========================================
# Cell 16: Summary
# ===========================================
print("\n" + "=" * 50)
print("MODELING COMPLETE")
print("=" * 50)
print(f"\n[Data]")
print(f"  Total samples: {len(df):,}")
print(f"  Train: {len(X_train):,}")
print(f"  Test: {len(X_test):,}")
print(f"\n[Models Trained]")
for r in all_results:
    print(f"  {r['model']}: RMSE={r['rmse']:.6f}, R2={r['r2']:.6f}, Time={r['train_time']:.2f}s")
print(f"\n[Best Model]")
print(f"  {best_model_name}")
print(f"\n[Output Files]")
print(f"  Models: {MODEL_PATH}")
print(f"  Results: {REPORT_PATH}")
print(f"\n[Next Step]")
print(f"  05_evaluation.ipynb - Detailed evaluation & visualization")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount done!
Project: EcoTracing
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 50.2 MB/s eta 0:00:00
Library load done!
Loaded parquet: 19,523,808 rows
Columns: ['machine_id', 'start_time_sec', 'end_time_sec', 'duration', 'hour', 'cpu', 'memory', 'power_w', 'energy_kwh']
Train: (15619046, 4)
Test: (3904762, 4)
Training LightGBM...
Time: 41.61s
RMSE: 0.000034
R2: 0.999970

Training XGBoost...
Time: 65.60s
RMSE: 0.000057
R2: 0.999918

Training Random Forest...
Time: 469.78s
RMSE: 0.000014
R2: 0.999995

Training CatBoost...
Time: 77.64s
RMSE: 0.000043
R2: 0.999953

MODEL COMPARISON SUMMARY
       model     rmse      mae       r2     mape  train_time
RandomForest 0.000014 0.000008 0.999995 0.066324  469.778683
    LightGBM 0.000034 0.000004 0.999970 0.15196